# 第13章　生成AIと大規模言語モデルの医療応用**『医療診断支援AIの社会実装（社会実装編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-social

## ハルシネーションを抑える設計 ― RAGと根拠の紐づけ

```text[検索でヒットした根拠]"（ここに、自施設の規程の該当箇所がそのまま引用される。上のガイドライン名・版・節を明記する）"[プロンプト]上記の根拠のみを用いて回答し、末尾に引用元(§)を必ず付す。根拠に無い数値・所見は述べないこと。該当が無ければ「該当なし」と答えよ。```

## マルチモーダルLLMは、どう画像を「読む」のか

In [ ]:
img_feat = vision_encoder(image)          # (パッチ数, d_v)img_tokens = projector(img_feat)          # LLMの埋め込み次元へ写像prompt_tokens = embed(text_prompt)        # 質問文の埋め込みseq = concat([img_tokens, prompt_tokens]) # 画像トークンを文脈に注入answer = llm.generate(seq)

## 画像側の基盤モデルと、自己教師あり事前学習の中身

In [ ]:
# InfoNCE の骨格：同一画像の2ビューを近づけ、他は遠ざけるz = F.normalize(encoder(torch.cat([view1, view2])), dim=1)   # 2N個の表現sim = z @ z.T / tau                                          # 全ペアの類似度sim.fill_diagonal_(-1e9)                                     # 自分自身は除外loss = F.cross_entropy(sim, positive_index)                 # 正例の位置を当てる